# Plotly + Cufflinks + Workalendar Demo

Small demo using `data_sales.csv` with interactive charts and business calendar enrichment.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import cufflinks as cf
from workalendar.europe import Spain

cf.go_offline()

possible_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_ROOT = None
for root in possible_roots:
    if (root / "09_Pandas").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root containing 09_Pandas folder.")

DATA_PATH = PROJECT_ROOT / "09_Pandas" / "data" / "data_sales.csv"
EXPORTS_DIR = PROJECT_ROOT / "09_Pandas" / "data" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["ventas"] = pd.to_numeric(df["ventas"], errors="coerce")
df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce")
df = df.dropna(subset=["fecha", "ventas", "cantidad"]).copy()
df["ingreso_total"] = df["ventas"] * df["cantidad"]
df.head()

## 1) Workalendar enrichment

In [ ]:
cal = Spain()
df["is_working_day"] = df["fecha"].apply(lambda d: cal.is_working_day(d.date()))
df["weekday"] = df["fecha"].dt.day_name()

working_day_summary = (
    df.groupby("is_working_day", as_index=False)["ingreso_total"]
    .sum()
    .rename(columns={"ingreso_total": "total_revenue"})
)
working_day_summary

## 2) Plotly interactive chart (monthly revenue)

In [ ]:
monthly = (
    df.assign(mes=df["fecha"].dt.to_period("M").astype(str))
    .groupby("mes", as_index=False)["ingreso_total"]
    .sum()
)

fig_monthly = px.bar(
    monthly,
    x="mes",
    y="ingreso_total",
    title="Monthly Revenue (Plotly)",
    labels={"mes": "Month", "ingreso_total": "Revenue"},
)
fig_monthly.show()

## 3) Cufflinks interactive chart (revenue by region)

In [ ]:
region_summary = (
    df.groupby("region", as_index=False)["ingreso_total"]
    .sum()
    .sort_values("ingreso_total", ascending=False)
)

region_summary.set_index("region")["ingreso_total"].iplot(
    kind="bar",
    title="Revenue by Region (Cufflinks)",
    xTitle="Region",
    yTitle="Revenue",
    color="#4C78A8",
)

## 4) Export enriched dataset

In [ ]:
output_path = EXPORTS_DIR / "sales_workalendar_enriched.csv"
df.to_csv(output_path, index=False)

print("Export generated:", output_path)
print("Rows:", len(df))
print("Working days count:", int(df["is_working_day"].sum()))
print("Non-working days count:", int((~df["is_working_day"]).sum()))